# Data Analysis

This notebook will contain descriptive and statistical analysis.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.paths import RAW_DIR, INTERIM_DIR, PROCESSED_DIR
from src.data_loader import load_data
from src.config import RANDOM_SEED

## I. Build the analysis table

## 1. Setup and parameter



In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)


CORE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]
DATE_COLS = CORE_COLS + ["order_estimated_delivery_date"]

SLOT_BINS = [-1, 7, 11, 17, 23]
SLOT_LABELS = [
    "Late night 00-08 (off)",
    "Morning 08-12 (in)",
    "Afternoon 12-18 (in)",
    "Evening 18-24 (off)",
]

PEAK_PERIODS = {
    "Black Friday week": ("2017-11-20", "2017-11-27"),   
}


HOLIDAYS = np.array(
    [f"{y}-{md}" for y in (2016, 2017, 2018) for md in ("01-01", "05-01", "12-25")],
    dtype="datetime64[D]",
)

# ---------- Phân khúc ----------
BULKY_KG = 15                                                
REMOTE_STATES = ["AC", "AM", "AP", "PA", "RO", "RR", "TO"]
SLOW_APPROVAL_DAYS = 1                                       
REGIME_CUTOFF = "2018-04-25"                                 

funnel = []

## 2. Load dữ liệu đã clean

In [ ]:
late = pd.read_csv(PROCESSED_DIR / "late_delivery_orders.csv", parse_dates=DATE_COLS)
items = pd.read_parquet(PROCESSED_DIR / "cleaned_order_items.parquet")

assert (late["order_status"] == "delivered").all(), "late chứa đơn không phải delivered"

funnel.append({"step": "delivered valid",
               "removed": 0, "remaining": len(late), "note": ""})

## 3. Single-item filtering

In [ ]:

assert all(late[c].dt.tz is None for c in DATE_COLS), "A date column is timezone-aware"

miss_core = late[CORE_COLS].isna().any(axis=1)
core = late[~miss_core]
note_missing = ", ".join(f"{int(late[c].isna().sum())} missing {c}" for c in CORE_COLS if late[c].isna().any())

neg_flags = pd.DataFrame({
    "approved_before_purchase": core["order_approved_at"] < core["order_purchase_timestamp"],
    "carrier_before_approved": core["order_delivered_carrier_date"] < core["order_approved_at"],
    "customer_before_carrier": core["order_delivered_customer_date"] < core["order_delivered_carrier_date"],
    "customer_before_purchase": core["order_delivered_customer_date"] < core["order_purchase_timestamp"],
})
neg_any = neg_flags.any(axis=1)
n_carrier_before_purchase = int(((core["order_delivered_carrier_date"] < core["order_purchase_timestamp"]) & neg_any).sum())
note_negative = ", ".join(f"{k}: {int(v)}" for k, v in neg_flags.sum().items() if v)
note_negative += f" (of which carrier before purchase: {n_carrier_before_purchase})"

anomaly_orders = pd.concat([
    late[miss_core].assign(drop_reason="missing_core_timestamp"),
    core[neg_any].assign(drop_reason="negative_interval"),
])
anomaly_orders["flag_carrier_before_purchase"] = (
    anomaly_orders["order_delivered_carrier_date"] < anomaly_orders["order_purchase_timestamp"]
).astype(int)

order_base = core[~neg_any].copy()

funnel.append({"step": "Drop orders missing core timestamps",
               "removed": len(late) - len(core), "remaining": len(core), "note": note_missing})
funnel.append({"step": "Drop orders with negative time intervals",
               "removed": len(core) - len(order_base), "remaining": len(order_base), "note": note_negative})

## 4.Gộp seller, tạo order_df

In [ ]:
items["product_weight_g"] = pd.to_numeric(items["product_weight_g"], errors="coerce")

seller_info = (
    items.groupby("order_id")
    .agg(
        seller_id=("seller_id", "first"),
        seller_state=("seller_state", "first"),
        n_sellers=("seller_id", "nunique"),
        product_category=("product_category_name", "first"),
        total_weight_g=("product_weight_g", lambda s: s.sum(min_count=1)),
        shipping_limit_date=("shipping_limit_date", "max"),
    )
    .reset_index()
)

order_df = order_base.merge(seller_info, on="order_id", how="left", validate="one_to_one")

## 5. Late order

In [ ]:
order_df["is_late_ts"] = order_df["is_late"].astype(int)
order_df["delay_days_ts"] = order_df["delay_days"]

days_over = (
    order_df["order_delivered_customer_date"].dt.normalize()
    - order_df["order_estimated_delivery_date"].dt.normalize()
).dt.days
order_df["is_late"] = (days_over > 0).astype(int)
order_df["delay_days"] = days_over.clip(lower=0)

## 6.Time slots across 3 axes

In [ ]:
ap = order_df["order_approved_at"]
pu = order_df["order_purchase_timestamp"]

order_df["approve_hour"] = ap.dt.hour
order_df["approve_slot"] = pd.cut(order_df["approve_hour"], SLOT_BINS, labels=SLOT_LABELS)
order_df["hours_type"] = np.where(order_df["approve_hour"].between(8, 17), "In-hours", "Off-hours")

order_df["weekday"] = ap.dt.day_name()
order_df["day_type"] = np.where(ap.dt.dayofweek >= 5, "Weekend", "Weekday")

order_df["month"] = ap.dt.to_period("M").astype(str)
season = pd.Series("Off-peak", index=order_df.index)
for name, (start, end) in PEAK_PERIODS.items():
    season[(ap >= start) & (ap < end)] = name
order_df["season"] = season
order_df["is_peak"] = (season != "Off-peak").astype(int)

order_df["purchase_hour"] = pu.dt.hour
order_df["purchase_slot"] = pd.cut(order_df["purchase_hour"], SLOT_BINS, labels=SLOT_LABELS)

ca = order_df["order_delivered_carrier_date"]
order_df["carrier_hour"] = ca.dt.hour
order_df["carrier_slot"] = pd.cut(order_df["carrier_hour"], SLOT_BINS, labels=SLOT_LABELS)
order_df["carrier_ts_regime"] = np.where(ca < REGIME_CUTOFF, "A: before 25/04/2018", "B: from 25/04/2018")

## 7.Duration of each stage

In [ ]:
def working_days(start, end):
    d0 = start.values.astype("datetime64[D]")
    d1 = end.values.astype("datetime64[D]")
    frac0 = ((start - start.dt.normalize()).dt.total_seconds() / 86400).values
    frac1 = ((end - end.dt.normalize()).dt.total_seconds() / 86400).values
    days = (
        np.busday_count(d0, d1, holidays=HOLIDAYS)
        - frac0 * np.is_busday(d0, holidays=HOLIDAYS)
        + frac1 * np.is_busday(d1, holidays=HOLIDAYS)
    )
    return pd.Series(days, index=start.index).clip(lower=0)

cu = order_df["order_delivered_customer_date"]
es = order_df["order_estimated_delivery_date"]

order_df["approval_days"] = (ap - pu).dt.total_seconds() / 86400
order_df["processing_lead_time"] = working_days(ap, ca)
order_df["carrier_transit_time"] = working_days(ca, cu)
order_df["lead_days"] = (cu - pu).dt.total_seconds() / 86400
order_df["estimated_lead_days"] = (es - pu).dt.total_seconds() / 86400

## 8.Seller delays and market segment shifts

In [ ]:
order_df["seller_delay"] = (order_df["order_delivered_carrier_date"] > order_df["shipping_limit_date"]).astype(int)

order_df["same_state"] = (order_df["customer_state"] == order_df["seller_state"]).astype(int)
order_df["route_type"] = np.where(order_df["same_state"] == 1, "Intra-state", "Inter-state")
order_df["route"] = order_df["seller_state"] + ">" + order_df["customer_state"]

order_df["is_multi_seller"] = (order_df["n_sellers"] > 1).astype(int)
order_df["is_bulky"] = (order_df["total_weight_g"] > BULKY_KG * 1000).astype(int)
order_df["is_remote"] = order_df["customer_state"].isin(REMOTE_STATES).astype(int)
order_df["slow_approval"] = (order_df["approval_days"] > SLOW_APPROVAL_DAYS).astype(int)
segment_summary = pd.DataFrame({
    "orders": [
        order_df["seller_delay"].sum(),
        order_df["same_state"].sum(),
        order_df["is_multi_seller"].sum(),
        order_df["is_bulky"].sum(),
        order_df["is_remote"].sum(),
        order_df["slow_approval"].sum(),
    ],
    "share_pct": [
        order_df["seller_delay"].mean() * 100,
        order_df["same_state"].mean() * 100,
        order_df["is_multi_seller"].mean() * 100,
        order_df["is_bulky"].mean() * 100,
        order_df["is_remote"].mean() * 100,
        order_df["slow_approval"].mean() * 100,
    ],
}, index=["seller_delay", "same_state", "is_multi_seller", "is_bulky", "is_remote", "slow_approval"]).round(2)
segment_summary

## 9.Save data

In [ ]:
interval_cols = ["approval_days", "processing_lead_time", "carrier_transit_time", "lead_days"]
assert (order_df[interval_cols] >= 0).all().all()

funnel_df = pd.DataFrame(funnel)
funnel_df.to_csv(TABLES_DIR / "step1_funnel.csv", index=False)

order_df.to_parquet(PROCESSED_DIR / "order_analysis.parquet", index=False)
anomaly_orders.to_parquet(PROCESSED_DIR / "anomaly_orders.parquet", index=False)
funnel_df


In [ ]:
print("order_df:", order_df.shape, "| anomaly_orders:", anomaly_orders.shape)
print("late rate:", round(order_df["is_late"].mean() * 100, 2), "| late rate (timestamp):", round(order_df["is_late_ts"].mean() * 100, 2))
print()
print(order_df["approve_slot"].value_counts().sort_index())
print()
print(order_df["season"].value_counts())
print()
print(order_df["carrier_ts_regime"].value_counts())
print()
print(order_df[["processing_lead_time", "carrier_transit_time"]].agg(["median", "mean"]).round(2))
print()
print("slow_approval:", int(order_df["slow_approval"].sum()), "| is_remote:", int(order_df["is_remote"].sum()))